# 04 - Early Warning Modeling (Nova Scotia Health System Pressure)

This notebook builds a simple predictive model to identify months of elevated EMS demand using upstream primary care access signals.

## Modeling goal

Assess whether prior-month changes in primary care access activity help anticipate downstream EMS system pressure

## Input dataset

Merged system signals dataset:

`data_processed/system_signals_merged_ns.csv`

## Target definition

We classify each zone-month as:

- **1 = High EMS demand month**
- **0 = Typical demand month**

High demand is defined as EMS responses above the zone-specific 75th percentile.

## Why this matters

If upstream primary care access changes precede EMS demand spikes, health planners could use these signals for early warning and resource planning.

In [1]:
from pathlib import Path
import pandas as pd

# Resolve project paths
cwd = Path.cwd()
project_root = cwd if (cwd / "data_processed").exists() else cwd.parent
data_processed_dir = project_root / "data_processed"
merged_path = data_processed_dir / "system_signals_merged_ns.csv"

print("Working directory:", cwd)
print("Project root:", project_root)
print("Merged file exists:", merged_path.exists(), "-", merged_path)

# Load merged modeling dataset
df = pd.read_csv(merged_path, parse_dates=["Date"])
print("Shape:", df.shape)
df.head()

Working directory: c:\ns-health-signals\notebooks
Project root: c:\ns-health-signals
Merged file exists: True - c:\ns-health-signals\data_processed\system_signals_merged_ns.csv
Shape: (128, 9)


,Zone,Date,pc_visits,pc_visits_roll3,pc_visits_pct_change,ehs_responses,ehs_responses_roll3,ehs_responses_pct_change,responses_zscore
0,Central,2021-04-01,926.0,769.333333,0.125152,3553.0,1948.500000,9.328488,-0.443670
1,Central,2021-05-01,534.0,761.000000,-0.423326,4805.0,2900.666667,0.352378,1.136497
2,Central,2021-06-01,800.0,753.333333,0.498127,3737.0,4031.666667,-0.222268,-0.211441
3,Central,2021-07-01,533.0,622.333333,-0.333750,3766.0,4102.666667,0.007760,-0.174840
4,Central,2021-08-01,684.0,672.333333,0.283302,4656.0,4053.000000,0.236325,0.948442


## Clean up leftover columns from earlier runs

In [2]:
ghost_cols = ["high_pressure", "predicted_high", "predicted_prob_high"]
existing_ghosts = [c for c in ghost_cols if c in df.columns]

if existing_ghosts:
    df = df.drop(columns=existing_ghosts)
    print("Dropped leftover columns:", existing_ghosts)
else:
    print("No leftover columns found.")

df.columns

No leftover columns found.


Index(['Zone', 'Date', 'pc_visits', 'pc_visits_roll3', 'pc_visits_pct_change',
       'ehs_responses', 'ehs_responses_roll3', 'ehs_responses_pct_change',
       'responses_zscore'],
      dtype='object')

## Creating 1-month lagged primary care features

In [3]:
# Ensure time order within each zone
df = df.sort_values(["Zone", "Date"]).copy()

# Prior-month primary care signals
df["pc_visits_lag1"] = df.groupby("Zone")["pc_visits"].shift(1)
df["pc_visits_roll3_lag1"] = df.groupby("Zone")["pc_visits_roll3"].shift(1)
df["pc_visits_pct_change_lag1"] = df.groupby("Zone")["pc_visits_pct_change"].shift(1)

# Remove first month per zone (lag undefined)
df = df.dropna(subset=[
    "pc_visits_lag1",
    "pc_visits_roll3_lag1",
    "pc_visits_pct_change_lag1"
]).copy()

print("Shape after lag features:", df.shape)
df.head()

Shape after lag features: (124, 12)


,Zone,Date,pc_visits,pc_visits_roll3,pc_visits_pct_change,ehs_responses,ehs_responses_roll3,ehs_responses_pct_change,responses_zscore,pc_visits_lag1,pc_visits_roll3_lag1,pc_visits_pct_change_lag1
1,Central,2021-05-01,534.0,761.000000,-0.423326,4805.0,2900.666667,0.352378,1.136497,926.0,769.333333,0.125152
2,Central,2021-06-01,800.0,753.333333,0.498127,3737.0,4031.666667,-0.222268,-0.211441,534.0,761.000000,-0.423326
3,Central,2021-07-01,533.0,622.333333,-0.333750,3766.0,4102.666667,0.007760,-0.174840,800.0,753.333333,0.498127
4,Central,2021-08-01,684.0,672.333333,0.283302,4656.0,4053.000000,0.236325,0.948442,533.0,622.333333,-0.333750
5,Central,2021-09-01,611.0,609.333333,-0.106725,3814.0,4078.666667,-0.180842,-0.114258,684.0,672.333333,0.283302


## Checking for feature ranges

In [4]:
# Quick check for extreme values
df[["pc_visits", "ehs_responses"]].describe()

# Show top/bottom pc_visits rows
df.sort_values("pc_visits").head(10)[["Zone","Date","pc_visits","ehs_responses"]], \
df.sort_values("pc_visits", ascending=False).head(10)[["Zone","Date","pc_visits","ehs_responses"]]

(        Zone       Date  pc_visits  ehs_responses
 7    Central 2022-03-01        1.0         3765.0
 104  Western 2022-01-01      102.0         2752.0
 106  Western 2022-03-01      201.0         2141.0
 105  Western 2022-02-01      224.0         2299.0
 6    Central 2021-10-01      282.0         4555.0
 32   Eastern 2021-07-01      303.0         1801.0
 30   Eastern 2021-05-01      328.0         2193.0
 33   Eastern 2021-08-01      354.0         2235.0
 35   Eastern 2021-10-01      358.0         2077.0
 31   Eastern 2021-06-01      384.0         1739.0,
         Zone       Date  pc_visits  ehs_responses
 28   Central 2023-12-01    21105.0         4492.0
 27   Central 2023-11-01    21089.0         3647.0
 25   Central 2023-09-01    20110.0         3751.0
 26   Central 2023-10-01    19199.0         4625.0
 24   Central 2023-08-01    18901.0         3628.0
 23   Central 2023-07-01    17572.0         4403.0
 22   Central 2023-06-01    17124.0         3518.0
 20   Central 2023-04-01    14

## Create target: high EMS demand month

We define high demand relative to each zone’s historical distribution to account for population differences.

In [5]:
zone_thresholds = df.groupby("Zone")["ehs_responses"].quantile(0.75)

df["high_ems_demand"] = df.apply(
    lambda r: 1 if r["ehs_responses"] >= zone_thresholds[r["Zone"]] else 0,
    axis=1
)

df["high_ems_demand"].value_counts()

high_ems_demand
0    93
1    31
Name: count, dtype: int64

## Check target balance and zone thresholds

In [6]:
zone_thresholds_df = zone_thresholds.rename("zone_75th").reset_index()
zone_thresholds_df

,Zone,zone_75th
0,Central,4507.75
1,Eastern,2053.75
2,Northern,2317.00
3,Western,2727.25


## Select modeling features

We use upstream primary care access indicators as predictors of EMS demand.

In [7]:
features = [
    "pc_visits_lag1",
    "pc_visits_roll3_lag1",
    "pc_visits_pct_change_lag1"
]

X = df[features]
y = df["high_ems_demand"]

X.head(), y.head()

(   pc_visits_lag1  pc_visits_roll3_lag1  pc_visits_pct_change_lag1
 1           926.0            769.333333                   0.125152
 2           534.0            761.000000                  -0.423326
 3           800.0            753.333333                   0.498127
 4           533.0            622.333333                  -0.333750
 5           684.0            672.333333                   0.283302,
 1    1
 2    0
 3    0
 4    1
 5    0
 Name: high_ems_demand, dtype: int64)

In [8]:
df.groupby("Zone")["high_ems_demand"].mean().rename("pct_high_ems_demand")

Zone
Central     0.25
Eastern     0.25
Northern    0.25
Western     0.25
Name: pct_high_ems_demand, dtype: float64

## Train/test split

We split the data randomly while preserving class balance.

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

len(X_train), len(X_test)

(93, 31)

## Train classification model

We use logistic regression for interpretability and small-dataset robustness.

In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight="balanced", max_iter=1000)
model.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :ter

## Evaluate model performance

In [11]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

# Inspect prediction distribution
pd.Series(y_pred, name="predicted_high_ems").value_counts()

Accuracy: 0.6129032258064516

Confusion matrix:
[[12 11]
 [ 1  7]]

              precision    recall  f1-score   support

           0       0.92      0.52      0.67        23
           1       0.39      0.88      0.54         8

    accuracy                           0.61        31
   macro avg       0.66      0.70      0.60        31
weighted avg       0.79      0.61      0.63        31



predicted_high_ems
1    18
0    13
Name: count, dtype: int64

## Final Evaluation summary

In [12]:
from sklearn.metrics import precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

print("Final evaluation metrics")
print("------------------------")
print(f"Accuracy:  {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall:    {recall:.2f}")
print(f"F1 score:  {f1:.2f}")

Final evaluation metrics
------------------------
Accuracy:  0.61
Precision: 0.39
Recall:    0.88
F1 score:  0.54


## Interpret model coefficients

Positive coefficients indicate higher likelihood of high EMS demand.

In [13]:
coef_table = pd.DataFrame({
    "feature": features,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)

coef_table

,feature,coefficient
2,pc_visits_pct_change_lag1,0.301168
0,pc_visits_lag1,0.000579
1,pc_visits_roll3_lag1,-0.000624


## Example early-warning interpretation

We examine months predicted as high EMS demand and compare upstream signals.

In [14]:
df["predicted_high_ems"] = model.predict(X)

cols_to_show = [
    "Zone", "Date",
    "predicted_high_ems", "high_ems_demand",
    "ehs_responses",
    "pc_visits_lag1", "pc_visits_roll3_lag1", "pc_visits_pct_change_lag1"
]
df.loc[df["predicted_high_ems"] == 1, cols_to_show].head(10)

,Zone,Date,predicted_high_ems,high_ems_demand,ehs_responses,pc_visits_lag1,pc_visits_roll3_lag1,pc_visits_pct_change_lag1
1,Central,2021-05-01,1,1,4805.0,926.0,769.333333,0.125152
3,Central,2021-07-01,1,0,3766.0,800.0,753.333333,0.498127
5,Central,2021-09-01,1,0,3814.0,684.0,672.333333,0.283302
9,Central,2022-05-01,1,1,4835.0,8994.0,3092.333333,8993.000000
10,Central,2022-06-01,1,0,3912.0,9228.0,6074.333333,0.026017
21,Central,2023-05-01,1,0,3471.0,14857.0,12755.333333,0.245243
23,Central,2023-07-01,1,0,4403.0,17124.0,15411.333333,0.201431
30,Eastern,2021-05-01,1,1,2193.0,422.0,312.666667,1.028846
32,Eastern,2021-07-01,1,0,1801.0,384.0,378.000000,0.170732
34,Eastern,2021-09-01,1,0,1769.0,354.0,347.000000,0.168317


## Key insight

Primary care access indicators show a measurable but limited association with downstream EMS demand across Nova Scotia health zones.

The model can identify some high-demand months but misses others, suggesting that primary care access dynamics alone do not fully explain EMS pressure variation.

This supports the value of combining multiple upstream and downstream indicators in future health system early-warning models.

In [15]:
df.to_csv("../data_processed/system_signals_modeling_ns.csv", index=False)

## Visualization

The merged and modeled system signals dataset was exported for visualization in Power BI.  
An interactive dashboard was created to illustrate the temporal relationship between prior-month primary care access activity and downstream EMS demand across Nova Scotia health zones.  

This visualization supports interpretation of the early-warning modeling results and highlights periods of elevated EMS demand relative to upstream access dynamics.

## Modeling Limitations

This modeling exercise has several limitations that should be considered when interpreting the results.

First, the dataset is relatively small and aggregated at the monthly health zone level. This limits the amount of variation available for training a predictive model and may reduce the stability of the estimates.

Second, the model only incorporates primary care access indicators as upstream predictors. Emergency Health Services demand is influenced by many additional factors such as seasonal illness patterns, demographic changes, weather events, and hospital capacity constraints that are not included in this dataset.

Third, lagged primary care indicators were used to approximate an early warning context. While this reflects a more realistic forecasting setup, these signals alone are unlikely to fully explain downstream EMS demand variation.

Finally, logistic regression was selected for interpretability and robustness with small datasets. More complex models could potentially improve predictive performance, but the goal of this analysis was to maintain transparency and clearly interpret the relationship between upstream access activity and EMS demand.

Future work could incorporate additional upstream indicators and external variables to build a more comprehensive health system pressure model.